# BTL số 1 – Antibandwidth Problem (ABP) bằng SAT Encoding
**Môn:** Các vấn đề hiện đại KHMT – HK I 2026–2027

Sườn notebook: mô hình SAT (PySAT) → các encoding AMO/ALO → solver đối chứng (CP-SAT, CPLEX CP/MIP, Gurobi) → thực nghiệm → phân tích.

Các chỗ đánh dấu `TODO` là phần bạn cần hoàn thiện.

## 0. Cài đặt & import

In [ ]:
# !pip install python-sat ortools pandas matplotlib
# !pip install gurobipy docplex   # cần license (Gurobi/CPLEX bản academic)

import time, random, itertools, math
import pandas as pd
import matplotlib.pyplot as plt

from pysat.formula import CNF, IDPool
from pysat.card import CardEnc, EncType
from pysat.solvers import Solver

TIMEOUT = 300      # giây cho mỗi lần chạy
SEED = 42
random.seed(SEED)

## 1. Bài toán & sinh instance
Cho đồ thị $G=(V,E)$, $|V|=n$. Tìm song ánh $f:V\to\{1..n\}$ **tối đa hóa** $\min_{(u,v)\in E}|f(u)-f(v)|$.

Bài toán quyết định: *có tồn tại $f$ với $|f(u)-f(v)|\ge k$ cho mọi cạnh?*

In [ ]:
def path_graph(n):  return n, [(i, i+1) for i in range(n-1)]
def cycle_graph(n): return n, [(i, (i+1) % n) for i in range(n)]

def grid_graph(r, c):
    idx = lambda i, j: i*c + j
    E = []
    for i in range(r):
        for j in range(c):
            if j+1 < c: E.append((idx(i,j), idx(i,j+1)))
            if i+1 < r: E.append((idx(i,j), idx(i+1,j)))
    return r*c, E

def random_graph(n, p, seed=SEED):
    rng = random.Random(seed)
    return n, [(u, v) for u in range(n) for v in range(u+1, n) if rng.random() < p]

# TODO: đọc instance benchmark từ thư mục instances/ (vd. Harwell-Boeing.).

## 2. SAT modeling & encoding
- Biến `x[v][i]`: đỉnh `v` nhận nhãn `i`.
- Mỗi đỉnh đúng 1 nhãn, mỗi nhãn đúng 1 đỉnh (ALO + AMO).
- Mỗi cạnh `(u,v)` và mọi cặp nhãn `|i-j| < k`: `¬x[u][i] ∨ ¬x[v][j]` (cấu trúc *staircase AMO*).

PySAT hỗ trợ sẵn: `pairwise`, `seqcounter`, `bitwise`, `ladder`, ... `commander` và `product` cần tự cài (TODO).

In [ ]:
AMO_ENCODINGS = {
    "pairwise":   EncType.pairwise,
    "seqcounter": EncType.seqcounter,
    "bitwise":    EncType.bitwise,
    "ladder":     EncType.ladder,
    # "commander": TODO tự cài
    # "product":   TODO tự cài
}

def build_cnf(n, edges, k, amo="seqcounter"):
    pool = IDPool()
    x = [[pool.id(("x", v, i)) for i in range(n)] for v in range(n)]
    cnf = CNF()
    enc = AMO_ENCODINGS[amo]

    # mỗi đỉnh đúng một nhãn
    for v in range(n):
        cnf.extend(CardEnc.equals(lits=x[v], bound=1, vpool=pool, encoding=enc).clauses)
    # mỗi nhãn đúng một đỉnh
    for i in range(n):
        cnf.extend(CardEnc.equals(lits=[x[v][i] for v in range(n)], bound=1,
                                  vpool=pool, encoding=enc).clauses)
    # ràng buộc khoảng cách trên cạnh
    for (u, v) in edges:
        for i in range(n):
            for j in range(max(0, i-k+1), min(n, i+k)):
                cnf.append([-x[u][i], -x[v][j]])
    return cnf, x, pool

# TODO: scl_staircase(n, edges, k) – SCL encoding theo paper thay cho vòng lặp cạnh ở trên

## 3. SAT solver: quyết định & tối ưu $k$

In [ ]:
def solve_decision(n, edges, k, amo="seqcounter", solver_name="cadical153"):
    t0 = time.time()
    cnf, x, pool = build_cnf(n, edges, k, amo)
    build_t = time.time() - t0
    with Solver(name=solver_name, bootstrap_with=cnf.clauses) as s:
        sat = s.solve()
        model = s.get_model() if sat else None
    label = None
    if sat:
        pos = set(l for l in model if l > 0)
        label = [next(i for i in range(n) if x[v][i] in pos) for v in range(n)]
    return sat, label, dict(vars=pool.top, clauses=len(cnf.clauses), build_time=build_t)

def maximize_k(n, edges, amo="seqcounter", solver_name="cadical153", timeout=TIMEOUT):
    # binary search: k=1 luôn khả thi (chỉ cần hoán vị bất kỳ)
    lo, hi, best = 1, n-1, None
    t0 = time.time()
    while lo <= hi and time.time() - t0 < timeout:
        mid = (lo + hi) // 2
        sat, label, stats = solve_decision(n, edges, mid, amo, solver_name)
        if sat: best, lo = (mid, label, stats), mid + 1
        else:   hi = mid - 1
    proven = lo > hi
    return dict(k=best[0] if best else None, label=best[1] if best else None,
                stats=best[2] if best else None, time=time.time()-t0, optimal=proven)

# Kiểm tra nhanh
n, E = cycle_graph(10)
print(maximize_k(n, E)["k"])

## 4. Solver đối chứng (CP-SAT, Gurobi, CPLEX CP/MIP)

In [ ]:
def solve_cpsat(n, edges, timeout=TIMEOUT):
    from ortools.sat.python import cp_model
    m = cp_model.CpModel()
    f = [m.NewIntVar(1, n, f"f{v}") for v in range(n)]
    m.AddAllDifferent(f)
    K = m.NewIntVar(1, n-1, "K")
    for (u, v) in edges:
        b = m.NewBoolVar("")
        m.Add(f[u] - f[v] >= K).OnlyEnforceIf(b)
        m.Add(f[v] - f[u] >= K).OnlyEnforceIf(b.Not())
    m.Maximize(K)
    s = cp_model.CpSolver()
    s.parameters.max_time_in_seconds = timeout
    t0 = time.time()
    st = s.Solve(m)
    return dict(k=int(s.ObjectiveValue()) if st in (cp_model.OPTIMAL, cp_model.FEASIBLE) else None,
                time=time.time()-t0, optimal=(st == cp_model.OPTIMAL))

def solve_gurobi(n, edges, timeout=TIMEOUT):
    # TODO: MIP – x[v][i] nhị phân, big-M cho |f(u)-f(v)| >= K
    raise NotImplementedError

def solve_cplex_mip(n, edges, timeout=TIMEOUT):
    # TODO: docplex.mp – mô hình tương tự Gurobi
    raise NotImplementedError

def solve_cplex_cp(n, edges, timeout=TIMEOUT):
    # TODO: docplex.cp – all_diff + abs(diff) >= K
    raise NotImplementedError

## 5. Thực nghiệm
Chạy mọi (instance × phương pháp), ghi CSV vào `results/`. Nhớ ghi lại cấu hình máy, phiên bản thư viện, timeout.

In [ ]:
instances = {
    "path_30":  path_graph(30),
    "cycle_30": cycle_graph(30),
    "grid_5x6": grid_graph(5, 6),
    "rand_40":  random_graph(40, 0.1),
    # TODO: thêm instance lớn (n = 100, 200, ...)
}

rows = []
for name, (n, E) in instances.items():
    for amo in AMO_ENCODINGS:
        for sname in ["cadical153", "glucose4"]:
            r = maximize_k(n, E, amo=amo, solver_name=sname)
            rows.append(dict(instance=name, n=n, m=len(E), method=f"SAT-{amo}-{sname}",
                             k=r["k"], time=r["time"], optimal=r["optimal"]))
    r = solve_cpsat(n, E)
    rows.append(dict(instance=name, n=n, m=len(E), method="CP-SAT",
                     k=r["k"], time=r["time"], optimal=r["optimal"]))
    # TODO: Gurobi, CPLEX CP, CPLEX MIP

df = pd.DataFrame(rows)
# df.to_csv("results/results.csv", index=False)
df

## 6. Phân tích kết quả

In [ ]:
pivot = df.pivot_table(index="instance", columns="method", values="time")
display(pivot.round(3))

pivot.T.plot(kind="bar", figsize=(12, 4), logy=True)
plt.ylabel("Thời gian (s)"); plt.title("Thời gian giải theo phương pháp")
plt.tight_layout()
# plt.savefig("results/time_comparison.png", dpi=200)
plt.show()

# TODO: bảng kích thước CNF (vars, clauses) theo encoding; biểu đồ scaling theo n

## 7. Kết luận & Data Availability
- **Kết luận:** encoding nào tốt nhất; SAT so với CP/MIP; ảnh hưởng của SCL.
- **Data Availability:** link GitHub repo chứa mã nguồn và kết quả thực nghiệm: `TODO`